## 13. Feature Engineering:
This is the most critical step of the pipeline. Algorithms are only as good as the math we feed them. We extract structural, non-linear signals from the raw price.

**Our Core Features:**
1. `log_return`: Normalizes the exponential compounding of stock prices.
2. `vol_6h` & `vol_24h`: We use **two** rolling windows to capture both intra-day flash crashes (6h) and multi-day sustained panics (24h).
3. `vwap_deviation`: Institutional traders buy near the VWAP. If the price drifts far from the VWAP, it signals an unnatural imbalance.
4. **CRITICAL - Avoiding Look-Ahead Bias:** When calculating a 24-hour rolling volatility, the first 23 rows mathematically cannot be computed. Using `fillna(0)` to fix this causes a *fatal error*. If we pad with zeros, the ML model will see a sequence of $0.00$ volatility and instantly classify the start of our dataset as an extreme anomaly because it's 'unnaturally' stable. Instead, we use `dropna()` to strictly drop the initial warmup rows, forcing the model to only evaluate truly valid math.

In [ ]:
# Notebook imports
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# Notebook configuration
FEATURE_COLS = [
    "log_return", "vol_6h", "vol_24h", "spread_pct", "vwap_deviation", "high_low_range"
],

In [ ]:
df = pd.read_csv(Path("C:/Users/hh/Market-Anomaly-Detection/data/processed/msft_hourly(in)_processed.csv"))

In [ ]:
# State check: requires cleaned dataframe from preprocessing
required = ["df"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing) +
        ". Run 1.0-abz-preprocessing.ipynb first in the same kernel."
    )

def require_columns(frame, cols, name="df"):
    missing_cols = [c for c in cols if c not in frame.columns]
    if missing_cols:
        raise RuntimeError(f"{name} missing columns: {', '.join(missing_cols)}")

require_columns(
    df,
    ["quote_datetime", "open", "high", "low", "close", "bid", "ask", "mid", "vwap"],
    name="df",
)
print("Prerequisites OK. Proceed with feature engineering.")


In [ ]:
def compute_features(data):
    features = data.copy()

    # Base transforms
    features["log_return"] = np.log(features["close"] / features["close"].shift(1))
    features["vol_6h"] = features["log_return"].rolling(6).std()
    features["vol_24h"] = features["log_return"].rolling(24).std()

    # Quotes & Execution
    features["spread_pct"] = (features["ask"] - features["bid"]) / features["mid"] * 100
    features["vwap_deviation"] = (features["close"] - features["vwap"]) / features["vwap"] * 100
    features["high_low_range"] = (features["high"] - features["low"]) / features["close"] * 100

    return features

# Compute and strictly drop NaNs
df_feat = compute_features(df)
print(f"Rows before dropping NaN warmup windows: {len(df_feat)}")
df_model = df_feat.dropna(subset=FEATURE_COLS).copy().reset_index(drop=True)
print(f"Rows strictly valid for ML modeling: {len(df_model)}")

In [ ]:
df_model.columns